<div style="display:block;width:100%;margin:auto;" direction=rtl align=center><br><br>
    <div  style="width:100%;margin:100;display:block;background-color:#fff0;"  display=block align=center>
        <table style="border-style:hidden;border-collapse:collapse;">             <tr>
                <td  style="border: none!important;">
                    <img width=130 align=right src="https://i.ibb.co/yXKQmtZ/logo1.png" style="margin:0;" />
                </td>
                <td style="text-align:center;border: none!important;">
                    <h1 align=center><font size=5 color="#045F5F"> <b>Armin Ghasemi</b><br><br>810100198</i></font></h1>
                </td>
                <td style="text-align:center;border: none!important;">
                    <h1 align=center><font size=5 color="#045F5F"> <b> ML </b><br><br>Final_Project</i></font></h1>
                </td>
                <td style="text-align:center;border: none!important;">
                    <h1 align=center><font size=5 color="#045F5F"> <b>  </b><br><br></i></font></h1>
                </td>
                <td style="border: none!important;">
                    <img width=170 align=left  src="https://i.ibb.co/wLjqFkw/logo2.png" style="margin:0;" />
                </td>
           </tr>
        </table>
    </div>
</div>

# 🎵 Audio Signal Processing & Feature Engineering
### Building the Dataset for Gender Classification & Speaker Recognition

> **University:** University of Tehran
> **Course:** Machine Learning
> **Project:** Final Capstone Project
> **Topic:** Audio Processing, MFCC Extraction, Data Engineering

---

## 📝 Project Overview
This notebook implements the **Preprocessing Pipeline** required to convert raw audio recordings into structured numerical datasets (CSVs) suitable for machine learning models. The pipeline transforms unstructured audio waves into feature vectors containing **MFCCs (Timbre)**, **F0 (Pitch)**, and **ZCR (Zero Crossing Rate)**.

**⚠️ Privacy & Data Note:**
The original dataset consisted of private voice recordings from course students. To protect privacy, **raw audio files are not included** in this repository.
* The code below demonstrates the exact logic used to generate the feature stores.
* To run this pipeline, please place your own `.wav` or `.mp3` files in the `data/raw_samples` directory.

**Key Objectives:**
1.  **Data Cleaning:** Organizing raw files into class-based directories (Male/Female).
2.  **Signal Processing (DSP):** Implementing Silence Removal (VAD), Low-pass Filtering, and Segmentation.
3.  **Feature Extraction:** Mathematical extraction of 20 MFCCs, Pitch, and ZCR.
4.  **Dataset Construction:** Generating the final `.csv` files used for clustering and classification tasks.

---

In [ ]:
import os

# ==========================================
# Global Configuration & Setup
# ==========================================
# This cell defines all project paths and signal processing constants.
# Run this cell first to ensure all directories are set up correctly.

# --- 1. Directory Structure Setup ---

# Get the current working directory of the notebook
BASE_DIR = os.getcwd()

# Define the root directory for all data-related files
DATA_ROOT = os.path.join(BASE_DIR, "data")

# Input Directory: Where raw audio files should be placed (e.g., .mp3, .wav)
# Expected Structure: ./data/raw_samples/male, ./data/raw_samples/female
RAW_AUDIO_DIR = os.path.join(DATA_ROOT, "raw_samples")

# Intermediate Directory: Where processed/segmented .wav files will be saved
PROCESSED_DIR = os.path.join(DATA_ROOT, "processed_segments")

# Output Directory: Where final extracted feature CSV files will be stored
FEATURES_DIR = os.path.join(DATA_ROOT, "feature_stores")

# --- 2. Directory Initialization ---

# Automatically create necessary directories if they don't exist to prevent errors
directories_to_create = [RAW_AUDIO_DIR, PROCESSED_DIR, FEATURES_DIR]

for directory in directories_to_create:
    os.makedirs(directory, exist_ok=True)
    # Optional: Print created paths to verify structure
    # print(f"Checked/Created directory: {directory}")

print(f"Project environment initialized at: {BASE_DIR}")
print("All necessary data directories are ready.")

# --- 3. Signal Processing Parameters ---

# These constants ensure consistency across all processing functions (Train & Test)
SAMPLE_RATE = 16000          # Target sampling rate (Hz) for librosa loading
SEGMENT_DURATION = 0.5       # Duration of each extracted audio segment (seconds)
NUM_SEGMENTS_TRAIN = 20      # Number of segments to extract per file for training data
NUM_SEGMENTS_TEST = 9        # Number of segments to extract per file for testing data

# Frequency limits for Pitch (F0) extraction
FMIN = librosa.note_to_hz('C2')
FMAX = librosa.note_to_hz('C7')

## 1. Data Organization
The raw data often comes in a mixed folder. The following script scans the filenames for keywords (e.g., "male", "female") and organizes them into structured subdirectories. This is the first step in creating labeled data for **Binary Classification**.

In [ ]:
import os
import shutil

def separate_audio_files(source_folder):
    """
    Scans the source folder and moves files into 'male' and 'female' subdirectories
    based on the filename keywords.
    """
    male_folder = os.path.join(source_folder, "male")
    female_folder = os.path.join(source_folder, "female")
    
    # Ensure subdirectories exist
    os.makedirs(male_folder, exist_ok=True)
    os.makedirs(female_folder, exist_ok=True)
    
    files_moved = 0
    for filename in os.listdir(source_folder):
        if filename.endswith(".mp3") or filename.endswith(".wav"):
            file_path = os.path.join(source_folder, filename)
            
            # Case-insensitive check for gender labels
            if "female" in filename.lower():
                shutil.move(file_path, os.path.join(female_folder, filename))
                files_moved += 1
            elif "male" in filename.lower():
                shutil.move(file_path, os.path.join(male_folder, filename))
                files_moved += 1

    print(f"Process complete. Total files separated: {files_moved}")

# --- Execution ---
# We use the RAW_AUDIO_DIR defined in Cell 1
if __name__ == "__main__":
    if os.path.exists(RAW_AUDIO_DIR):
        separate_audio_files(RAW_AUDIO_DIR)
    else:
        print(f"Warning: Directory not found: {RAW_AUDIO_DIR}")

In [ ]:
import os
import shutil
import random

def select_random_files(source_folder, num_files=50):
    """
    Randomly selects a subset of files from the source folder and moves them
    to a 'selected_files' subdirectory.
    """
    destination_folder = os.path.join(source_folder, "selected_files")
    os.makedirs(destination_folder, exist_ok=True)
    
    # Filter only files (exclude directories)
    all_files = [f for f in os.listdir(source_folder) if os.path.isfile(os.path.join(source_folder, f))]
    
    # Select random sample
    selected_files = random.sample(all_files, min(num_files, len(all_files)))
    
    for file in selected_files:
        shutil.move(os.path.join(source_folder, file), os.path.join(destination_folder, file))
    
    print(f"Moved {len(selected_files)} random files from: {source_folder}")

# --- Execution ---
# Paths are constructed dynamically based on the raw directory
male_source = os.path.join(RAW_AUDIO_DIR, "male")
female_source = os.path.join(RAW_AUDIO_DIR, "female")

if os.path.exists(female_source):
    select_random_files(female_source)

if os.path.exists(male_source):
    select_random_files(male_source)

## 2. Digital Signal Processing (DSP) Pipeline
Before feature extraction, raw audio signals must be cleaned and standardized. We apply the following transformations:

* **Low-Pass Filter (Butterworth):** Removes high-frequency noise (>4000Hz) irrelevant to human speech.
* **Voice Activity Detection (VAD):** Uses an energy-based threshold to remove silence and keep only active speech segments.
* **Segmentation:** Splits long audio files into uniform **0.5-second chunks** to increase the number of training samples.
* **Windowing:** Applies a Hamming Window to reduce spectral leakage.

In [ ]:
import os
import librosa
import numpy as np
import scipy.signal as signal
import soundfile as sf

def process_audio_files(input_folder, output_base_folder, class_name, target_sr=SAMPLE_RATE, num_segments=NUM_SEGMENTS_TRAIN, segment_duration=SEGMENT_DURATION):
    """
    Applies filters, removes silence, and segments audio files.
    Saves the processed segments into a specific output directory.
    """
    # Create specific output folder (e.g., data/processed_segments/train_male)
    output_folder = os.path.join(output_base_folder, f"train_{class_name}")
    os.makedirs(output_folder, exist_ok=True)
    
    file_counter = 1
    processed_count = 0
    
    for filename in os.listdir(input_folder):
        if filename.endswith(".mp3") or filename.endswith(".wav"):
            file_path = os.path.join(input_folder, filename)
            
            try:
                # 1. Load Audio
                audio, sr = librosa.load(file_path, sr=target_sr)
                
                # 2. Low-pass Filter (Butterworth)
                nyquist = target_sr / 2
                cutoff_freq = 4000
                b, a = signal.butter(6, cutoff_freq / nyquist, btype='low')
                filtered_audio = signal.filtfilt(b, a, audio)
                
                # 3. Voice Activity Detection (Energy-based)
                energy = np.abs(filtered_audio)
                threshold = np.percentile(energy, 75)
                speech_indices = np.where(energy > threshold)[0]
                
                # 4. Segmentation
                segment_length = int(target_sr * segment_duration)
                
                for _ in range(num_segments):
                    if len(speech_indices) < segment_length:
                        break
                        
                    # Random start point from active speech regions
                    start_idx = np.random.choice(speech_indices[:-segment_length])
                    segment = filtered_audio[start_idx:start_idx + segment_length]
                    
                    # 5. Windowing & Normalization
                    windowed_segment = segment * np.hamming(len(segment))
                    normalized_segment = windowed_segment / (np.max(np.abs(windowed_segment)) + 1e-6)
                    
                    # Save
                    output_path = os.path.join(output_folder, f"{class_name}_{file_counter}.wav")
                    sf.write(output_path, normalized_segment, target_sr)
                    file_counter += 1
                
                processed_count += 1
                
            except Exception as e:
                print(f"Error processing {filename}: {e}")

    print(f"Processing complete for {class_name}. Processed {processed_count} files.")

# --- Execution ---
# Define input paths (assuming separated folders exist)
path_train_male = os.path.join(RAW_AUDIO_DIR, "male")
path_train_female = os.path.join(RAW_AUDIO_DIR, "female")

# Process if directories exist
if os.path.exists(path_train_male):
    process_audio_files(path_train_male, PROCESSED_DIR, "male")

if os.path.exists(path_train_female):
    process_audio_files(path_train_female, PROCESSED_DIR, "female")

## 3. Feature Extraction
In this step, we convert the processed `.wav` segments into numerical vectors. For each audio segment, we extract a 23-dimensional feature vector:

| Feature | Count | Description |
| :--- | :--- | :--- |
| **MFCCs** | 20 | Mel-Frequency Cepstral Coefficients. Represents the "timbre" or shape of the vocal tract. |
| **F0 (Pitch)** | 1 | The fundamental frequency. Crucial for distinguishing male vs. female voices. |
| **ZCR** | 1 | Zero-Crossing Rate. Indicates the rate of sign-changes in the signal (measure of "noisiness"). |
| **Label** | 1 | `0` for Female, `1` for Male. |

In [ ]:
# === New Cell: Feature Extraction Function Definition (Train) ===
import os
import librosa
import numpy as np
import csv

def extract_features(folder_path, output_csv, label_value, target_sr=SAMPLE_RATE):
    """
    Extracts MFCCs, F0 (Pitch), and ZCR features from processed audio segments.
    Saves the result to a CSV file.
    Designed for Training Data (Single folder -> Single CSV).
    """
    data = []
    
    print(f"Extracting features from: {folder_path}")
    
    # Check if folder exists
    if not os.path.exists(folder_path):
        print(f"Error: Folder not found -> {folder_path}")
        return

    files = [f for f in os.listdir(folder_path) if f.endswith('.wav')]
    
    if not files:
        print("No .wav files found in this directory.")
        return

    for i, filename in enumerate(files):
        file_path = os.path.join(folder_path, filename)
        
        try:
            audio, sr = librosa.load(file_path, sr=target_sr)
            
            # MFCC Extraction
            mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20)
            mfccs_mean = np.mean(mfccs, axis=1)
            
            # Pitch (F0) Extraction
            f0, voiced_flag, _ = librosa.pyin(audio, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'))
            f0_mean = np.mean(f0[voiced_flag]) if np.any(voiced_flag) else 0
            
            # Zero Crossing Rate
            zcr = librosa.feature.zero_crossing_rate(audio)
            zcr_mean = np.mean(zcr)
            
            # Construct Feature Vector
            feature_vector = list(mfccs_mean) + [f0_mean, zcr_mean, label_value]
            data.append([filename] + feature_vector)
            
        except Exception as e:
            print(f"Error extracting features for {filename}: {e}")
            
        if i % 100 == 0 and i > 0:
            print(f"Processed {i} files...")

    # Save to CSV
    header = ["filename"] + [f"MFCC_{i+1}" for i in range(20)] + ["F0", "ZCR", "label"]
    
    with open(output_csv, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(header)
        writer.writerows(data)
    
    print(f"Successfully saved features to {output_csv}")

In [ ]:
# --- Execution for Train Data (Male & Female) ---

# 1. Process Female Data
processed_female_path = os.path.join(PROCESSED_DIR, "train_female")
output_csv_female = os.path.join(FEATURES_DIR, "gender_train_female.csv")

if os.path.exists(processed_female_path):
    extract_features(processed_female_path, output_csv_female, label_value=0)
else:
    print(f"Directory not found: {processed_female_path}")

# 2. Process Male Data
processed_male_path = os.path.join(PROCESSED_DIR, "train_male")
output_csv_male = os.path.join(FEATURES_DIR, "gender_train_male.csv")

if os.path.exists(processed_male_path):
    extract_features(processed_male_path, output_csv_male, label_value=1)
else:
    print(f"Directory not found: {processed_male_path}")

## 4. Test Data Processing (Speaker Identification Setup)
For the **Speaker Identification** task, knowing only the gender is not enough. We need to track the identity of each speaker.
Therefore, the processing logic changes slightly: instead of dumping all files into one folder, we create **unique subfolders** for each speaker (e.g., `SPEAKER_1`, `SPEAKER_2`). This preserves the identity structure needed for multi-class classification.

In [ ]:
import os
import librosa
import numpy as np
import scipy.signal as signal
import soundfile as sf

def process_test_files(folder_path, output_base_folder, prefix, target_sr=SAMPLE_RATE, num_segments=NUM_SEGMENTS_TEST, segment_duration=SEGMENT_DURATION):
    """
    Processes test files. Creates a unique subfolder for each speaker/file 
    to facilitate Speaker Identification tasks.
    """
    # Base folder for test segments (e.g., data/processed_segments/test_male)
    test_output_root = os.path.join(output_base_folder, f"test_{prefix}")
    os.makedirs(test_output_root, exist_ok=True)
    
    speaker_counter = 1
    
    for filename in os.listdir(folder_path):
        if filename.endswith(".mp3") or filename.endswith(".wav"):
            file_path = os.path.join(folder_path, filename)
            
            # Create a separate folder for each speaker (TEST1, TEST2, ...)
            speaker_folder = os.path.join(test_output_root, f"{prefix}_SPEAKER_{speaker_counter}")
            os.makedirs(speaker_folder, exist_ok=True)
            speaker_counter += 1
            
            try:
                # Load and Filter (Same logic as training)
                audio, sr = librosa.load(file_path, sr=target_sr)
                
                nyquist = target_sr / 2
                b, a = signal.butter(6, 4000 / nyquist, btype='low')
                filtered_audio = signal.filtfilt(b, a, audio)
                
                energy = np.abs(filtered_audio)
                threshold = np.percentile(energy, 75)
                speech_indices = np.where(energy > threshold)[0]
                
                segment_length = int(target_sr * segment_duration)
                file_counter = 1
                
                for _ in range(num_segments):
                    if len(speech_indices) < segment_length:
                        break
                    start_idx = np.random.choice(speech_indices[:-segment_length])
                    segment = filtered_audio[start_idx:start_idx + segment_length]
                    
                    # Windowing
                    windowed_segment = segment * np.hamming(len(segment))
                    normalized_segment = windowed_segment / (np.max(np.abs(windowed_segment)) + 1e-6)
                    
                    output_path = os.path.join(speaker_folder, f"seg_{file_counter}.wav")
                    sf.write(output_path, normalized_segment, target_sr)
                    file_counter += 1
                    
            except Exception as e:
                print(f"Error processing {filename}: {e}")
    
    print(f"Test data processing complete for {prefix}.")

# --- Execution ---
# Define input paths for test data
test_male_raw = os.path.join(RAW_AUDIO_DIR, "test_male")
test_female_raw = os.path.join(RAW_AUDIO_DIR, "test_female")

if os.path.exists(test_male_raw):
    process_test_files(test_male_raw, PROCESSED_DIR, "male")

if os.path.exists(test_female_raw):
    process_test_files(test_female_raw, PROCESSED_DIR, "female")

In [ ]:
import os
import librosa
import numpy as np
import csv

def extract_features_recursive(root_folder, label_value, target_sr=SAMPLE_RATE):
    """
    Recursively iterates through speaker subfolders, extracts features, 
    and creates a CSV for each subfolder.
    """
    
    for subfolder in os.listdir(root_folder):
        subfolder_path = os.path.join(root_folder, subfolder)
        
        # Check if it's a directory (e.g., male_SPEAKER_1)
        if os.path.isdir(subfolder_path):
            data = []
            output_csv = os.path.join(subfolder_path, f"features_{subfolder}.csv")
            
            for filename in os.listdir(subfolder_path):
                if filename.endswith(".wav"):
                    file_path = os.path.join(subfolder_path, filename)
                    
                    try:
                        audio, sr = librosa.load(file_path, sr=target_sr)
                        
                        # Features
                        mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20)
                        mfccs_mean = np.mean(mfccs, axis=1)
                        
                        f0, voiced_flag, _ = librosa.pyin(audio, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'))
                        f0_mean = np.mean(f0[voiced_flag]) if np.any(voiced_flag) else 0
                        
                        zcr = librosa.feature.zero_crossing_rate(audio)
                        zcr_mean = np.mean(zcr)
                        
                        feature_vector = list(mfccs_mean) + [f0_mean, zcr_mean, label_value]
                        data.append([filename] + feature_vector)
                        
                    except Exception as e:
                        pass # Skip erroneous files
            
            # Save individual CSV
            if data:
                header = ["filename"] + [f"MFCC_{i+1}" for i in range(20)] + ["F0", "ZCR", "label"]
                with open(output_csv, mode='w', newline='') as file:
                    writer = csv.writer(file)
                    writer.writerow(header)
                    writer.writerows(data)
                print(f"Generated features for: {subfolder}")

# --- Execution ---
processed_test_female = os.path.join(PROCESSED_DIR, "test_female")
processed_test_male = os.path.join(PROCESSED_DIR, "test_male")

if os.path.exists(processed_test_female):
    extract_features_recursive(processed_test_female, label_value=0)

if os.path.exists(processed_test_male):
    extract_features_recursive(processed_test_male, label_value=1)

## 5. Data Aggregation
The final step collects all the individual CSV files generated from subfolders and consolidates them into the final dataset structure stored in `data/feature_stores/`. These are the files that will be loaded in the Clustering and Classification notebooks.

In [ ]:
import os
import shutil

def collect_csv_files(source_root, dest_folder, prefix):
    """
    Crawls through the source directory, finds all CSV files, 
    and moves them to a centralized destination folder.
    Renames files if duplicates exist.
    """
    if not os.path.exists(dest_folder):
        os.makedirs(dest_folder)
        
    print(f"Collecting CSVs from {source_root} to {dest_folder}...")

    count = 0
    for root, dirs, files in os.walk(source_root):
        for file in files:
            if file.endswith('.csv'):
                file_path = os.path.join(root, file)
                
                # Create a unique name to avoid overwriting (e.g. features_SPEAKER_1.csv)
                output_path = os.path.join(dest_folder, file)
                
                # Handle duplicate names
                counter = 1
                name, ext = os.path.splitext(file)
                while os.path.exists(output_path):
                    output_path = os.path.join(dest_folder, f"{name}_{counter}{ext}")
                    counter += 1
                
                shutil.copy(file_path, output_path)
                count += 1

    print(f"Finished. Collected {count} CSV files for {prefix}.")

# --- Execution ---
# Define destinations for organized CSVs
dest_csv_male = os.path.join(FEATURES_DIR, "speaker_id_male")
dest_csv_female = os.path.join(FEATURES_DIR, "speaker_id_female")

processed_test_male = os.path.join(PROCESSED_DIR, "test_male")
processed_test_female = os.path.join(PROCESSED_DIR, "test_female")

if os.path.exists(processed_test_male):
    collect_csv_files(processed_test_male, dest_csv_male, "Male")

if os.path.exists(processed_test_female):
    collect_csv_files(processed_test_female, dest_csv_female, "Female")